# 05 — Net-of-Cost IC

L1 LOB quote join → spread_bps → net IC. Coverage fraction always reported.
5 bps fallback used only when lob_matched=False.

In [1]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.join_lob_quotes import join_l1_quotes
from src.costs import run_net_of_cost, net_ic
from src.stats_rigor import bootstrap_ic_ci
from src.grid import append_cells
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2016, LOB_DIRS

setup_style()
N_BOOT   = 1000
HORIZONS = [1, 5, 15]
print('Setup complete.')

Setup complete.


## 1. Load and join L1 quotes for AAPL (smoke-test ticker)

In [2]:
aapl_path = PANELS_2016.get('AAPL')
lob_dir   = LOB_DIRS.get('AAPL', '')

if not aapl_path or not Path(aapl_path).exists():
    raise RuntimeError(f'AAPL 2016-2020 panel not found at {aapl_path}')
if not lob_dir or not Path(lob_dir).exists():
    raise RuntimeError(f'AAPL LOB directory not found: {lob_dir}')

panel = pd.read_csv(aapl_path)
panel['stock'] = 'AAPL'
print(f'Loaded panel: {len(panel):,} rows')

panel_lob = join_l1_quotes(panel, lob_dir=lob_dir)

coverage = panel_lob['lob_matched'].mean() if 'lob_matched' in panel_lob.columns else np.nan
print(f'LOB coverage (lob_matched): {coverage:.1%}')
if 'spread_bps' in panel_lob.columns:
    print(f'Median spread (matched rows): {panel_lob.loc[panel_lob["lob_matched"], "spread_bps"].median():.2f} bps')

Loaded panel: 6,252 rows


  [join_lob] Coverage: 6252/6252 events (100.0%) matched to a LOB quote.
LOB coverage (lob_matched): 100.0%
Median spread (matched rows): 0.48 bps


## 2. Net IC across horizons

In [3]:
panel_lob['ofi_x_llm'] = panel_lob['ofi_z'] * panel_lob['llm_score']

net_rows = []
for h in HORIZONS:
    ret_col = f'ret_{h}m'
    if ret_col not in panel_lob.columns:
        continue
    res = net_ic(panel_lob, horizon=h)
    # gross IC from bootstrap
    gross = bootstrap_ic_ci(panel_lob['ofi_x_llm'], panel_lob[ret_col], n_boot=N_BOOT, seed=42)
    net_rows.append({
        'horizon': h,
        'gross_ic': gross['ic'],
        'gross_ci_lo': gross['ci_lo'],
        'gross_ci_hi': gross['ci_hi'],
        'net_ic': res.get('net_ic', np.nan),
        'mean_cost_bps': res.get('mean_cost_bps', np.nan),
        'coverage_frac': res.get('coverage_frac', np.nan),
        'n': gross['n'],
    })

cost_tbl = pd.DataFrame(net_rows)
print(cost_tbl.to_string(index=False, float_format='{:.4f}'.format))

 horizon  gross_ic  gross_ci_lo  gross_ci_hi  net_ic  mean_cost_bps  coverage_frac    n
       1    0.0108      -0.0134       0.0348 -0.2041         3.3723         1.0000 6234
       5   -0.0043      -0.0283       0.0191 -0.1194         3.3867         1.0000 6191
      15   -0.0020      -0.0264       0.0219 -0.0807         3.4250         1.0000 6076


## 3. Multi-ticker net IC (where LOB available)

In [4]:
multi_rows = []
for ticker, path in PANELS_2016.items():
    lob_dir_t = LOB_DIRS.get(ticker, '')
    if not lob_dir_t or not Path(lob_dir_t).exists():
        print(f'  skip {ticker}: no LOB dir')
        continue
    if not Path(path).exists():
        print(f'  skip {ticker}: panel not found')
        continue
    try:
        df = pd.read_csv(path)
        df['stock'] = ticker
        df_lob = join_l1_quotes(df, lob_dir=lob_dir_t)
        df_lob['ofi_x_llm'] = df_lob['ofi_z'] * df_lob['llm_score']
        cov = df_lob['lob_matched'].mean() if 'lob_matched' in df_lob.columns else np.nan
        for h in HORIZONS:
            ret_col = f'ret_{h}m'
            if ret_col not in df_lob.columns:
                continue
            res = net_ic(df_lob, horizon=h)
            gross = bootstrap_ic_ci(df_lob['ofi_x_llm'], df_lob[ret_col], n_boot=N_BOOT, seed=42)
            multi_rows.append({
                'ticker': ticker, 'horizon': h,
                'gross_ic': gross['ic'],
                'net_ic':   res.get('net_ic', np.nan),
                'coverage_frac': cov,
                'mean_cost_bps': res.get('mean_cost_bps', np.nan),
                'n': gross['n'],
            })
    except Exception as e:
        print(f'  ERROR {ticker}: {e}')

if multi_rows:
    multi_tbl = pd.DataFrame(multi_rows)
    print(multi_tbl.to_string(index=False, float_format='{:.4f}'.format))
    save_table(multi_tbl, '05_net_ic_multi',
               caption='Net-of-cost IC (ofi\_x\_llm) per ticker. coverage\_frac = fraction of events with matched L1 LOB quote; 5 bps round-trip fallback applied to unmatched rows.',
               label='tab:net_ic')

  [join_lob] Coverage: 6252/6252 events (100.0%) matched to a LOB quote.


  [join_lob] Coverage: 660/660 events (100.0%) matched to a LOB quote.


  [join_lob] Coverage: 2443/2443 events (100.0%) matched to a LOB quote.


  [join_lob] Coverage: 431/431 events (100.0%) matched to a LOB quote.


  [join_lob] Coverage: 2153/2153 events (100.0%) matched to a LOB quote.


  [join_lob] Coverage: 517/517 events (100.0%) matched to a LOB quote.


ticker  horizon  gross_ic  net_ic  coverage_frac  mean_cost_bps    n
  AAPL        1    0.0108 -0.2041         1.0000         3.3723 6234
  AAPL        5   -0.0043 -0.1194         1.0000         3.3867 6191
  AAPL       15   -0.0020 -0.0807         1.0000         3.4250 6076
   AMD        1    0.0558 -0.3107         1.0000         7.1137  660
   AMD        5    0.0075 -0.1790         1.0000         7.1393  656
   AMD       15    0.0455 -0.0869         1.0000         7.0619  649
   JPM        1   -0.0083 -0.2649         1.0000         3.7714 2440
   JPM        5   -0.0104 -0.1461         1.0000         3.7834 2428
   JPM       15    0.0066 -0.0794         1.0000         3.8239 2383
    MU        1    0.0025 -0.2440         1.0000         3.4204  431
    MU        5   -0.0525 -0.1580         1.0000         3.4162  426
    MU       15   -0.0225 -0.0877         1.0000         3.3979  417
  NFLX        1   -0.0161 -0.5165         1.0000        35.6136 2148
  NFLX        5   -0.0186 -0.3483 

## 4. Gross vs net figure

In [5]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(HORIZONS))

gross_ics = cost_tbl['gross_ic'].values
net_ics   = cost_tbl['net_ic'].values
lo = gross_ics - cost_tbl['gross_ci_lo'].values
hi = cost_tbl['gross_ci_hi'].values - gross_ics

ax.bar(x - 0.2, gross_ics, width=0.35, label='Gross IC',
       yerr=[lo, hi], error_kw={'linewidth': 0.8, 'capsize': 4},
       color='#2E86AB')
ax.bar(x + 0.2, net_ics, width=0.35, label='Net IC', color='#A23B72', alpha=0.8)
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'{h}-min' for h in HORIZONS])
ax.set_ylabel('Spearman IC')
ax.set_title(f'AAPL: Gross vs net IC (LOB coverage {coverage:.0%})')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)
save_fig(fig, '05_gross_vs_net_ic')
plt.close()

# Save AAPL table
save_table(cost_tbl, '05_aapl_net_ic',
           caption=f'AAPL gross vs net IC (LOB coverage={coverage:.1%}). Net IC uses L1 spread for matched rows; 5 bps fallback for unmatched.',
           label='tab:aapl_net_ic')

  Saved figure → results/figures/05_gross_vs_net_ic.pdf
  Saved table  → results/tables/05_aapl_net_ic.csv + results/tables/05_aapl_net_ic.tex


PosixPath('results/tables/05_aapl_net_ic.csv')

## 5. Append to BH grid

In [6]:
grid_rows = []
for _, row in cost_tbl.iterrows():
    grid_rows.append({
        'notebook': '05', 'cell_id': f"net_aapl_{row['horizon']}m",
        'stock': 'AAPL', 'scorer': 'ofi_x_llm|net',
        'horizon': row['horizon'], 'n': row['n'],
        'ic': row['net_ic'], 'ci_lo': np.nan, 'ci_hi': np.nan,
        'p': np.nan,
    })
append_cells(grid_rows)
print(f'Appended {len(grid_rows)} cells to secondary grid.')

Appended 3 cells to secondary grid.
